## nb_02_landing_to_bronze
Download files from kaggle to landing zone and ingest them into bronze schema
Since we are just downloading and loading into bronze schema a single notebook will do
as they handle the same datasets.

### Load functions for landing and bronze activities

In [15]:
%run nb_02_landing_bronze_functions

StatementMeta(, d8c77771-d46a-4095-8e78-bf5e3dbea706, 66, Finished, Available, Finished, True)

### Imports

In [16]:
from typing import Dict, List, Optional

import os
import kagglehub
import pandas as pd
from pyspark.sql import Row, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

StatementMeta(, d8c77771-d46a-4095-8e78-bf5e3dbea706, 67, Finished, Available, Finished, False)

### Define variables

In [17]:
WORKSPACE: str = "JDE10"
LAKEHOUSE: str = "NHL_db"

ABFSS_BASE_PATH: str = f"abfss://{WORKSPACE}@onelake.dfs.fabric.microsoft.com"
LANDING_PATH: str = f"{ABFSS_BASE_PATH}/{LAKEHOUSE}.Lakehouse/Files/landing"


StatementMeta(, d8c77771-d46a-4095-8e78-bf5e3dbea706, 68, Finished, Available, Finished, False)

### Run: Kaggle → landing

In [18]:
configs = get_active_configs(spark)
datasets_needed = group_by_kaggle_dataset(configs)

failed_landings: List[str] = []

for kaggle_dataset, rows in datasets_needed.items():
    print(f"⬇️ Downloading dataset: {kaggle_dataset}")
    try:
        download_path = download_kaggle_dataset(kaggle_dataset)
    except RuntimeError as e:
        print(f"❌ {e}")
        failed_landings.extend(row.bronze_table_name for row in rows)
        continue
    print(f"Dataset downloaded to: {download_path}")

    available_files = build_file_lookup(download_path)

    for row in rows:
        source_file = row.source_file
        table_name = row.bronze_table_name

        if source_file not in available_files:
            print(f"⚠️ {source_file} not found for table '{table_name}' — skipping.")
            failed_landings.append(table_name)
            continue

        local_file = available_files[source_file]
        if not save_file_to_landing(local_file, source_file, table_name, LANDING_PATH):
            failed_landings.append(table_name)

if failed_landings:
    print(f"\n⚠️ {len(failed_landings)} table(s) failed to land: {failed_landings}")
else:
    print("\n🏁 Landing zone load complete.")

# Fail loudly only if EVERYTHING failed — a partial landing still lets
# nb_01's own bronze step (and downstream silver notebooks) proceed for
# whatever did land. Tighten this to `if failed_landings:` if you'd rather
# treat any single failed table as fatal.
if len(failed_landings) == len(configs):
    raise RuntimeError("All configured datasets failed to land — aborting before bronze load.")

StatementMeta(, d8c77771-d46a-4095-8e78-bf5e3dbea706, 69, Finished, Available, Finished, False)

⬇️ Downloading dataset: martinellis/nhl-game-data
Dataset downloaded to: /home/trusted-service-user/.cache/kagglehub/datasets/martinellis/nhl-game-data/versions/2
✅ Saved game_skater_stats.csv → landing/game_skater_stats.csv (for table 'bronze.game_skater_stats')
✅ Saved game_goalie_stats.csv → landing/game_goalie_stats.csv (for table 'bronze.game_goalie_stats')
✅ Saved player_info.csv → landing/player_info.csv (for table 'bronze.player_info')
✅ Saved game_teams_stats.csv → landing/game_teams_stats.csv (for table 'bronze.game_teams_stats')
✅ Saved team_info.csv → landing/team_info.csv (for table 'bronze.team_info')
✅ Saved game.csv → landing/game.csv (for table 'bronze.game')

🏁 Landing zone load complete.


### Run: landing → bronze

In [20]:
# Re-query in case config_datasets changed since the landing step above.
# LANDING_PATH is reused from the earlier cell in this notebook's session.
configs = get_active_configs(spark)

for row in configs:
    table_name = row.bronze_table_name
    hash_columns = row.hash_columns
    csv_option = row.csv_option or None
    source_file = row.source_file

    file_path = f"{LANDING_PATH}/{source_file}"
    print(f"\n📥 Loading {source_file} → {table_name}")

    # preflight checks for content change
    result = has_file_content_changed(spark, file_path)
    if not result["changed"]:
        print(f"Skipping (no change): {file_path}")

    else: 
        print(f"Ingesting: {file_path}")

        result = (
            ingest_to_bronze(
                spark,
                file_path,
                table_name,
                hash_columns=hash_columns,
                csv_options=csv_option,
            )
        )        

print("\n🏁 Bronze load complete.")

StatementMeta(, d8c77771-d46a-4095-8e78-bf5e3dbea706, 71, Finished, Available, Finished, False)


📥 Loading game_skater_stats.csv → bronze.game_skater_stats
Skipping (no change): abfss://JDE10@onelake.dfs.fabric.microsoft.com/NHL_db.Lakehouse/Files/landing/game_skater_stats.csv

📥 Loading game_goalie_stats.csv → bronze.game_goalie_stats
Skipping (no change): abfss://JDE10@onelake.dfs.fabric.microsoft.com/NHL_db.Lakehouse/Files/landing/game_goalie_stats.csv

📥 Loading player_info.csv → bronze.player_info
Ingesting: abfss://JDE10@onelake.dfs.fabric.microsoft.com/NHL_db.Lakehouse/Files/landing/player_info.csv
[bronze.player_info] Rows read from 'abfss://JDE10@onelake.dfs.fabric.microsoft.com/NHL_db.Lakehouse/Files/landing/player_info.csv': 1
[bronze.player_info] Merge complete. Inserted 1 new row(s). Total rows now: 3926.

📥 Loading game_teams_stats.csv → bronze.game_teams_stats
Skipping (no change): abfss://JDE10@onelake.dfs.fabric.microsoft.com/NHL_db.Lakehouse/Files/landing/game_teams_stats.csv

📥 Loading team_info.csv → bronze.team_info
Skipping (no change): abfss://JDE10@onelake